# 02 - GRPO on a mock tool environment

Continues from the SFT adapter of `01_sft.ipynb`. The planner now learns from its own rollouts: for each question it
runs up to three rounds of real tool calls against a local environment, and a rule-based reward scores the outcome.
Training uses TRL's `GRPOTrainer` with native multi-turn tool calling, as in Unsloth's *Qwen3 (4B) GRPO Tool Use*
notebook: tool results are masked out of the loss, and a tool error is sent back to the model instead of crashing.

In order:

1. **BFCL** on the SFT checkpoint.
2. Load the model with Unsloth's vLLM engine and the SFT adapter.
3. **GRPO**, checkpointing to Google Drive.
4. **PlannerQA-Bench** on the GRPO checkpoint; push adapter, merged weights and GGUF.
5. **BFCL** on the GRPO checkpoint, and a final base / SFT / GRPO comparison.

Use the same configuration as in `01_sft.ipynb` (same `SMOKE`, `MODEL_KEY`), so both notebooks share one Drive folder.
After a disconnect: *Run all* again.

## 1. Configuration

In [ ]:
SMOKE = True                 # must match 01_sft.ipynb
MODEL_KEY = "qwen3-1.7b"     # must match 01_sft.ipynb
ENABLE_THINKING = False
REWARD_MODE = "graded_cost"  # "graded_cost" (default) | "graded" | "binary"
USE_DRIVE = True
RESUME = True
RUN_BFCL = True
WANDB_PROJECT = "tool-use-planner"
PUSH_TO_HUB = True
HF_PRIVATE = False
PUSH_GGUF = True
OVERRIDES = {}               # e.g. {"grpo_steps": 200}; see planner/config.py

## 2. Install and set up

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    if is_t4:
        # Keep the machine's torch: upgrading it can pull a CUDA build the driver cannot load
        try:
            import importlib.metadata as _md
            _torch, _tv = (f'{_p}=={_md.version(_p).split("+")[0]}' for _p in ('torch', 'torchvision'))
        except Exception:
            _torch, _tv = 'torch', 'torchvision'
        !uv pip install -qqq {_numpy} {_pil} {_torch} {_tv} bitsandbytes xformers unsloth
    else:
        !uv pip install -qqq --upgrade vllm==0.19.1 {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq triton
    try:
        import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
    except Exception:
        _torch_v = ()
    _torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
    !uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install transformers==5.15.1 "datasets>=4.7.0,<5.0.0"
!uv pip install --no-deps trl==1.13.0
!uv pip install -qqq --no-deps --force-reinstall "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo" "unsloth @ git+https://github.com/unslothai/unsloth"
!uv pip install -qqq bm25s==0.3.16 wandb

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/yy726/tool_use_demo.git"
REPO_BRANCH = "claude/inspiring-planck-c4a3hh"

if os.path.isdir("planner"):                      # already inside a checkout
    REPO_DIR = os.getcwd()
elif os.path.isdir("../planner"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/tool_use_demo"
    if not os.path.isdir(REPO_DIR):
        url = REPO_URL
        try:                                      # private repo: add a GITHUB_TOKEN secret
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            if token: url = url.replace("https://", f"https://{token}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
print("repo:", REPO_DIR)

In [ ]:
OVERRIDES = dict(OVERRIDES, reward_mode=REWARD_MODE)
os.environ.setdefault("UNSLOTH_DECODE_CUDA_GRAPHS", "0")   # eager decoding: the CUDA-graph decode path held ~30 GB outside PyTorch and ran an A100-40GB out of memory
import unsloth  # import before transformers / trl so its patches apply
import json, random
from planner.config import get_config
from planner.persist import StageCache, persist_dir, read_secrets
from planner.setup_env import prepare

cfg = get_config("smoke" if SMOKE else "a100", MODEL_KEY, enable_thinking=ENABLE_THINKING, **OVERRIDES)

print("secrets:", read_secrets())                 # HF_TOKEN, WANDB_API_KEY, GITHUB_TOKEN from Colab Secrets
HF_USER = None
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login, whoami
    login(token=os.environ["HF_TOKEN"])
    HF_USER = whoami()["name"]

PERSIST = persist_dir(cfg.exp_name, USE_DRIVE)    # Google Drive: survives a disconnect
LOCAL = f"/content/work/{cfg.exp_name}"            # local disk: large merged weights
os.makedirs(LOCAL, exist_ok=True)
CACHE = StageCache(PERSIST, enabled=RESUME)
CACHE.check_config(cfg.fingerprint())
print("persist dir:", PERSIST, "| finished stages:", CACHE.done())

env = prepare(cfg, os.path.join(PERSIST, "env"))  # world + knowledge base + sft / grpo / eval tasks

## 3. BFCL on the SFT checkpoint

Run now, while the GPU is free. Base-model scores come from the cache written by `01_sft.ipynb`.

In [ ]:
from planner import bfcl
from planner.unsloth_utils import resolve_merged

SFT_LORA = os.path.join(PERSIST, "sft_lora")
SFT_MERGED = os.path.join(LOCAL, "sft_merged")
STAGE = f"grpo_{cfg.reward_mode}"                  # reward ablations get their own folders and cache entries
GRPO_DIR = os.path.join(PERSIST, STAGE)
GRPO_LORA = os.path.join(PERSIST, f"{STAGE}_lora")
GRPO_MERGED = os.path.join(LOCAL, f"{STAGE}_merged")
HUB_STAGE = "grpo" if cfg.reward_mode == "graded_cost" else STAGE.replace("_", "-")
assert os.path.exists(os.path.join(SFT_LORA, "adapter_model.safetensors")), "Run 01_sft.ipynb first (same SMOKE / MODEL_KEY)."

BFCL = {"base": bfcl.run_stage(CACHE, "base", cfg, PERSIST, enabled=RUN_BFCL)}
merged = resolve_merged(cfg, "sft", SFT_MERGED, HF_USER) if RUN_BFCL and not CACHE.has("bfcl_sft") else None
BFCL["sft"] = bfcl.run_stage(CACHE, "sft", cfg, PERSIST, local_model_path=merged, enabled=RUN_BFCL)
# A GRPO checkpoint that is finished but not scored yet (see the last section) is scored now, while the GPU is free.
if CACHE.has(f"bfcl_{STAGE}"):
    BFCL["grpo"] = CACHE.load(f"bfcl_{STAGE}")
elif RUN_BFCL and os.path.exists(os.path.join(GRPO_LORA, "adapter_model.safetensors")):
    BFCL["grpo"] = bfcl.run_stage(CACHE, STAGE, cfg, PERSIST, local_model_path=resolve_merged(cfg, HUB_STAGE, GRPO_MERGED, HF_USER), enabled=True)
shown = {k: v for k, v in BFCL.items() if v}
if shown: print(bfcl.format_table(shown))

## 4. Load the model and the SFT adapter

`fast_inference=True` gives Unsloth's vLLM engine, which generates the rollouts with the LoRA being trained.

In [ ]:
from unsloth import FastLanguageModel
from planner.bench import format_report, run_benchmark, text_backend
from planner.chat import setup_chat_format
from planner.tasks import build_prompt
from planner.unsloth_utils import has_vllm, hf_generate_fn, load_lora_weights, load_model, vllm_generate_fn

MAX_NEW = cfg.bench_max_new_tokens * (4 if cfg.enable_thinking else 1)

def bench(name, lora_dir=None):
    """PlannerQA-Bench for the current weights (or a saved LoRA when the vLLM engine is on). Cached."""
    def run():
        if FAST:
            generate = vllm_generate_fn(model, tokenizer, lora_dir, MAX_NEW)
        else:
            FastLanguageModel.for_inference(model)
            generate = hf_generate_fn(model, tokenizer, MAX_NEW, cfg.bench_batch)
        return run_benchmark(env.tasks["eval"], text_backend(fmt.render_prompt, generate), batch_size=cfg.bench_batch, progress=True)
    return CACHE.get_or_run(f"bench_{name}", run)

FAST = has_vllm()                                  # no vLLM on a T4: rollouts fall back to transformers (slow)
model, tokenizer = load_model(cfg, fast_inference=FAST, gpu_memory_utilization=0.6)
fmt = setup_chat_format(tokenizer, enable_thinking=cfg.enable_thinking)
load_lora_weights(model, SFT_LORA)

BENCH = {"base": CACHE.load("bench_base") if CACHE.has("bench_base") else None, "sft": bench("sft", SFT_LORA)}
BENCH = {k: v for k, v in BENCH.items() if v}
print(format_report(BENCH))

## 5. The environment, the tools and the reward

The four tools are plain Python functions with type hints and docstrings; TRL derives their schemas and runs them
inside the rollout loop. `knowledge_search` is BM25 over a local Wikipedia-paragraph index and stands in for the
expensive retrieval tool (cost 1.0 per call; the others cost 0.25).

The reward needs no LLM judge. Every training task carries gold labels: the answer, and the evidence a correct plan
must retrieve. Evidence is read only from tool messages, which Python writes, so the model cannot fake it.

```
g = (min(relevance, factuality) - 1) / 3                       # 0, 1/3, 2/3 or 1
R = g - 0.3 * min(1, excess_cost / 3) - 0.2 * invalid_fraction   # "graded_cost"
```

`excess_cost` is the cost above the task's cheapest correct plan: necessary calls are free, redundant ones are not.
Taking the minimum of the two axes means guessing an answer without evidence earns at most 1/3, and retrieving
evidence without answering earns 0.

In [ ]:
from planner.rewards import ToolLedger, make_reward_fn
from planner.scoring import score_messages
from planner.tasks import grpo_rows
from planner.tools import MAX_TOOL_ROUNDS, TOOLS, execute_tool

task = next(t for t in env.tasks["grpo"] if t["family"] == "tool_bridge")
print("Q:", task["query"])
for calls in task["optimal_plan"]:
    for c in calls: print(" ", c["tool"], c["args"], "->", execute_tool(c["tool"], c["args"])[:160])
print("gold answer:", task["aspects"][0]["answers"], "| cheapest plan costs", task["optimal_cost"])

## 6. GRPO

Each optimizer step uses `grpo_batch * grpo_accum / grpo_num_generations` prompts with `grpo_num_generations` rollouts
each. Rollouts stop when the model answers without a tool call, after three tool rounds, or when
`max_completion_length` (which counts tool results too) is used up. No KL term (`beta = 0`).

Logged to W&B every step: the reward, good rate, both axes, tool cost, the share of groups whose rollouts all got the
same reward (those carry no learning signal), TRL's `tools/call_frequency` and `tools/failure_frequency`, and a running
count of calls per tool under `qps/` - an estimate of what this run would cost against real tools.

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from planner.persist import latest_checkpoint, log_summary, wandb_init
from planner.unsloth_utils import filter_kwargs, save_adapter

os.makedirs(GRPO_DIR, exist_ok=True)
wandb_run, REPORT_TO = wandb_init(GRPO_DIR, WANDB_PROJECT, f"{STAGE}-{cfg.repo_stem}", cfg.to_dict())
ledger = ToolLedger(os.path.join(GRPO_DIR, "tool_ledger.json"))

if RESUME and os.path.exists(os.path.join(GRPO_LORA, "adapter_model.safetensors")):
    load_lora_weights(model, GRPO_LORA)
    print("GRPO already finished: adapter loaded from Drive")
else:
    FastLanguageModel.for_training(model)
    dataset = Dataset.from_list(grpo_rows(env.tasks["grpo"], cfg.seed))
    extra = {"chat_template_kwargs": {"enable_thinking": cfg.enable_thinking}} if fmt.is_thinking_model else {}
    args = GRPOConfig(**filter_kwargs(GRPOConfig, dict(
        temperature = 1.0,
        learning_rate = cfg.grpo_lr,
        weight_decay = 0.001,
        warmup_ratio = 0.1,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        beta = 0.0,
        logging_steps = 1,
        per_device_train_batch_size = cfg.grpo_batch,
        gradient_accumulation_steps = cfg.grpo_accum,
        num_generations = cfg.grpo_num_generations,
        max_completion_length = cfg.grpo_max_completion,
        max_tool_calling_iterations = MAX_TOOL_ROUNDS,
        max_steps = cfg.grpo_steps,
        save_steps = cfg.grpo_save_steps,
        save_total_limit = 2,
        seed = cfg.seed,
        report_to = REPORT_TO,
        run_name = f"{STAGE}-{cfg.repo_stem}",
        output_dir = GRPO_DIR,
        **extra,
    )))
    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [make_reward_fn(cfg.reward_mode, ledger = ledger, print_every = 20)],
        tools = TOOLS,
        args = args,
        train_dataset = dataset,
    )
    checkpoint = latest_checkpoint(GRPO_DIR) if RESUME else None
    print("resume from checkpoint:", checkpoint)
    trainer.train(resume_from_checkpoint = checkpoint)
    save_adapter(model, tokenizer, fmt, GRPO_LORA)
print("tool calls made during RL:", dict(ledger.calls), "| rollouts:", ledger.rollouts, "| total cost:", round(ledger.total_cost, 1))

## 7. Evaluate the GRPO model

In [ ]:
BENCH["grpo"] = bench(STAGE, GRPO_LORA)
print(format_report(BENCH))
for name, report in BENCH.items(): log_summary(wandb_run, f"bench_{name}", report["summary"])

## 8. Save and push

In [ ]:
from planner.persist import model_card
from planner.unsloth_utils import export_model

card = model_card(
    f"{cfg.repo_stem}-{HUB_STAGE}",
    cfg.base_model_hf,
    f"Tool-calling planner after SFT and GRPO (reward: `{cfg.reward_mode}`). It plans with `knowledge_search`, `calculator`, "
    "`get_weather` and `get_election` for at most three tool rounds and ends with an `Answer:` line. Code: " + REPO_URL,
    {"PlannerQA-Bench (held-out)": format_report(BENCH), "BFCL": bfcl.format_table({k: v for k, v in BFCL.items() if v}) if any(BFCL.values()) else "not run"},
)
status = export_model(model, tokenizer, fmt, cfg, HUB_STAGE, HF_USER, GRPO_LORA, GRPO_MERGED, card, push = PUSH_TO_HUB, private = HF_PRIVATE, gguf = PUSH_GGUF)
print(status)

## 9. BFCL on the GRPO checkpoint, and the final comparison

BFCL needs most of the GPU for its own vLLM server. The cell first tries to release the training model. If the GPU is
still busy it says so: then choose *Runtime > Restart session* and *Run all*. Every finished stage is skipped, and the
BFCL cell in section 3 scores the GRPO checkpoint.

In [ ]:
from planner.unsloth_utils import gpu_free_fraction, release_gpu

if RUN_BFCL and not BFCL.get("grpo") and has_vllm():
    try: del trainer
    except NameError: pass
    del model
    free = release_gpu()
    if free >= 0.7:
        BFCL["grpo"] = bfcl.run_stage(CACHE, STAGE, cfg, PERSIST, local_model_path=GRPO_MERGED, enabled=True, gpu_memory_utilization=round(0.9 * free, 2))
    else:
        print(f"Only {free:.0%} of the GPU is free. Restart the session and Run all: section 3 will score the GRPO checkpoint.")

report = "## PlannerQA-Bench (800 held-out tasks)\n\n" + format_report(BENCH)
shown = {k: v for k, v in BFCL.items() if v}
if shown:
    report += "\n\n## BFCL (official checker, offline categories)\n\n" + bfcl.format_table(shown)
    for name, result in shown.items(): log_summary(wandb_run, f"bfcl_{name}", dict(result["groups"], bfcl_core=result["bfcl_core"]))
report += f"\n\n## Tool calls made during RL\n\n{dict(ledger.calls)} over {ledger.rollouts} rollouts (cost {ledger.total_cost:.1f})\n"
with open(os.path.join(PERSIST, f"report_{STAGE}.md"), "w") as f: f.write(report)
if wandb_run: wandb_run.finish()

from IPython.display import Markdown, display
display(Markdown(report))

### Reading the result

- **SFT** is expected to move BFCL and the simpler task families: it teaches formats, schemas and the tool inventory
  with dense per-token supervision.
- **GRPO** is expected to move the hard families (`kb_bridge`, `tool_bridge`, `composite`) and the tool cost, because it
  trains on the states the model itself reaches and optimises the outcome directly. On BFCL it should hold steady: the
  GRPO environment has a fixed tool set, so BFCL here is a check that general function calling was not lost.
- The failure-tag table shows which error types each stage removed.

These are expectations, not guarantees: compare them with the tables above. To try a different reward, change
`REWARD_MODE` and run again - SFT results and the environment are reused.